# Locker Placement Pattern Analysis

**Goal:** Analyze bpost's 2,379 existing parcel locker locations to identify what makes a good placement site. Train an XGBoost binary classifier that distinguishes locker sites from non-locker sites, then extract feature importance rankings and placement profiles per zone type.

**Data sources:**
- Local: bbox.json (lockers), centroids.json, sectors.json, competitors.json, competitive_coverage.json, strategic_quadrants.json, demand_scores.json, supermarkets.json
- External: Overpass API (~200 tile-batched queries), Regional zoning WFS/WMS APIs (~2,400 queries)

**Approach:**
1. Enrich each locker with local features (demographics, competition, proximity) — zero API calls
2. Enrich with OSM data via Overpass tile batching — ~200 API calls
3. Enrich with urbanism/zoning data via regional APIs — ~2,400 API calls
4. Build negative samples (hard, random, near-miss)
5. Train XGBoost with spatial cross-validation (GroupKFold by municipality)
6. Output SHAP feature importance and placement profiles

## Section 0: Setup & Imports

In [ ]:
import json
import math
import os
import time
import warnings
from collections import defaultdict
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import shap
import xgboost as xgb
from shapely.geometry import Point
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore", category=FutureWarning)
plt.style.use("seaborn-v0_8-whitegrid")

# Paths
ROOT = Path("..").resolve()
DATA = ROOT / "data"
ENRICHED = DATA / "enriched"
TRAINING = DATA / "training"
OUTPUT = Path("output")

for d in [ENRICHED, TRAINING, OUTPUT]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Root: {ROOT}")
print(f"Data: {DATA}")
print(f"Enriched cache: {ENRICHED}")
print(f"Training output: {TRAINING}")
print(f"Notebook output: {OUTPUT}")

In [ ]:
# --- Spatial utility functions (adapted from scripts/local_analysis.py) ---

R_EARTH = 6_371_000  # meters
DEG_TO_RAD = math.pi / 180
CELL_SIZE = 0.01  # degrees (~1.1 km)


def haversine(lat1, lng1, lat2, lng2):
    """Distance in meters between two WGS84 points."""
    d_lat = (lat2 - lat1) * DEG_TO_RAD
    d_lng = (lng2 - lng1) * DEG_TO_RAD
    a = (math.sin(d_lat / 2) ** 2 +
         math.cos(lat1 * DEG_TO_RAD) * math.cos(lat2 * DEG_TO_RAD) *
         math.sin(d_lng / 2) ** 2)
    return R_EARTH * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def build_spatial_index(points, key_lat="lat", key_lng="lng"):
    """Grid-based spatial index for fast radius queries."""
    index = defaultdict(list)
    for i, p in enumerate(points):
        lat = p[key_lat] if isinstance(p, dict) else getattr(p, key_lat)
        lng = p[key_lng] if isinstance(p, dict) else getattr(p, key_lng)
        k = (int(lat // CELL_SIZE), int(lng // CELL_SIZE))
        index[k].append(i)
    return index


def find_within_radius(index, points, lat, lng, radius_m,
                       key_lat="lat", key_lng="lng"):
    """Find all points within radius_m of (lat, lng). Returns [(idx, dist)]."""
    radius_deg = radius_m / 111_000
    cell_r = math.ceil(radius_deg / CELL_SIZE)
    base_lat = int(lat // CELL_SIZE)
    base_lng = int(lng // CELL_SIZE)
    results = []
    for dlat in range(-cell_r, cell_r + 1):
        for dlng in range(-cell_r, cell_r + 1):
            k = (base_lat + dlat, base_lng + dlng)
            for idx in index.get(k, []):
                p = points[idx]
                plat = p[key_lat] if isinstance(p, dict) else getattr(p, key_lat)
                plng = p[key_lng] if isinstance(p, dict) else getattr(p, key_lng)
                d = haversine(lat, lng, plat, plng)
                if d <= radius_m:
                    results.append((idx, d))
    return results


def count_within_radius(index, points, lat, lng, radius_m,
                        key_lat="lat", key_lng="lng"):
    """Count points within radius_m."""
    return len(find_within_radius(index, points, lat, lng, radius_m,
                                  key_lat, key_lng))


def nearest_distance(index, points, lat, lng, max_radius=50_000,
                     key_lat="lat", key_lng="lng", exclude_idx=None):
    """Distance to the nearest point (meters). Returns inf if none found."""
    hits = find_within_radius(index, points, lat, lng, max_radius,
                              key_lat, key_lng)
    if exclude_idx is not None:
        hits = [(i, d) for i, d in hits if i != exclude_idx]
    if not hits:
        return float("inf")
    return min(d for _, d in hits)


print("Spatial utilities ready.")

In [ ]:
# --- Load all data files ---

with open(DATA / "bbox.json") as f:
    lockers_raw = json.load(f)
print(f"Lockers: {len(lockers_raw)}")

with open(DATA / "centroids.json") as f:
    centroids_raw = json.load(f)
centroids_map = {c["sc"]: c for c in centroids_raw}
print(f"Sector centroids: {len(centroids_raw)}")

with open(DATA / "competitors.json") as f:
    competitors_raw = json.load(f)
print(f"Competitors: {len(competitors_raw)}")

with open(DATA / "supermarkets.json") as f:
    supermarkets_raw = json.load(f)
print(f"Supermarkets: {len(supermarkets_raw)}")

with open(DATA / "competitive_coverage.json") as f:
    comp_cov = json.load(f)
comp_cov_sectors = comp_cov["sectors"]
print(f"Competitive coverage sectors: {len(comp_cov_sectors)}")

with open(DATA / "strategic_quadrants.json") as f:
    quadrants = json.load(f)
quadrants_sectors = quadrants["sectors"]
print(f"Strategic quadrant sectors: {len(quadrants_sectors)}")

with open(DATA / "demand_scores.json") as f:
    demand_raw = json.load(f)
demand_map = {d["sc"]: d for d in demand_raw}
print(f"Demand scores: {len(demand_raw)}")

# Sectors GeoDataFrame (for point-in-polygon and adjacency)
print("Loading sectors GeoJSON (18 MB)...")
sectors_gdf = gpd.read_file(DATA / "sectors.json")
print(f"Sectors GeoDataFrame: {len(sectors_gdf)} features")

# Region code mapping
RGN_MAP = {"04000": "Brussels-Capital", "03000": "Wallonia", "02000": "Flanders"}
sectors_gdf["region"] = sectors_gdf["rgn"].map(RGN_MAP)
print(f"Region distribution:\n{sectors_gdf['region'].value_counts().to_string()}")

## Section 1: Local Feature Enrichment (zero API calls)

Assign each locker to its statistical sector via point-in-polygon, then enrich with:
- Sector demographics (pop, density, demand, age ratio, income index)
- Competition metrics (competitor count, own coverage, gap)
- Strategic quadrant
- Proximity counts (competitors, supermarkets, other lockers at 500m / 1km / 2km)

In [ ]:
CACHE_LOCAL = ENRICHED / "lockers_local_features.parquet"

if CACHE_LOCAL.exists():
    print(f"Loading cached local features from {CACHE_LOCAL}")
    lockers_df = pd.read_parquet(CACHE_LOCAL)
    print(f"  {len(lockers_df)} lockers loaded with {len(lockers_df.columns)} features")
else:
    # --- 1a. Point-in-polygon: assign each locker to a sector ---
    print("1a. Point-in-polygon assignment...")
    locker_points = gpd.GeoDataFrame(
        lockers_raw,
        geometry=[Point(l["lng"], l["lat"]) for l in lockers_raw],
        crs="EPSG:4326",
    )

    # Spatial join: assign each locker its sector properties
    lockers_joined = gpd.sjoin(locker_points, sectors_gdf, how="left", predicate="within")

    # Deduplicate: lockers on sector boundaries may match multiple sectors
    if lockers_joined.index.duplicated().any():
        n_dupes = lockers_joined.index.duplicated().sum()
        print(f"  Deduplicating {n_dupes} boundary matches (keeping first)")
        lockers_joined = lockers_joined[~lockers_joined.index.duplicated(keep="first")]

    # Handle lockers that fell outside all polygons (edge cases near borders)
    unmatched = lockers_joined["sc"].isna().sum()
    if unmatched > 0:
        print(f"  {unmatched} lockers unmatched — using nearest sector")
        unmatched_idx = lockers_joined[lockers_joined["sc"].isna()].index
        for idx in unmatched_idx:
            pt = lockers_joined.loc[idx, "geometry"]
            dists = sectors_gdf.geometry.distance(pt)
            nearest_idx = dists.idxmin()
            for col in sectors_gdf.columns:
                if col != "geometry":
                    lockers_joined.loc[idx, col] = sectors_gdf.loc[nearest_idx, col]

    print(f"  {len(lockers_joined)} lockers assigned to sectors")
    print(f"  Sector columns: {list(sectors_gdf.columns[:12])}")

    # --- 1b. Enrich with sector-level features ---
    print("1b. Enriching with sector-level features...")

    # Demographics from centroids
    lockers_joined["demand"] = lockers_joined["sc"].map(
        lambda sc: centroids_map.get(sc, {}).get("demand", 0))
    lockers_joined["ageRatio"] = lockers_joined["sc"].map(
        lambda sc: centroids_map.get(sc, {}).get("ageRatio", 0))
    lockers_joined["incomeIdx"] = lockers_joined["sc"].map(
        lambda sc: centroids_map.get(sc, {}).get("incomeIdx", 0))

    # Competition metrics
    lockers_joined["cc"] = lockers_joined["sc"].map(
        lambda sc: comp_cov_sectors.get(sc, {}).get("cc", 0))
    lockers_joined["oc"] = lockers_joined["sc"].map(
        lambda sc: comp_cov_sectors.get(sc, {}).get("oc", 0))
    lockers_joined["gap"] = lockers_joined["sc"].map(
        lambda sc: comp_cov_sectors.get(sc, {}).get("gap", 1.0))
    lockers_joined["num_operators"] = lockers_joined["sc"].map(
        lambda sc: len(comp_cov_sectors.get(sc, {}).get("ops", [])))

    # Strategic quadrant
    lockers_joined["quadrant"] = lockers_joined["sc"].map(
        lambda sc: quadrants_sectors.get(sc, "unknown"))

    # Municipality from demand scores (for GroupKFold later)
    lockers_joined["municipality"] = lockers_joined["sc"].map(
        lambda sc: demand_map.get(sc, {}).get("mun", "unknown"))

    # --- 1c. Proximity features ---
    print("1c. Computing proximity features...")

    # Build spatial indices
    comp_idx = build_spatial_index(competitors_raw)
    super_idx = build_spatial_index(supermarkets_raw)
    locker_idx = build_spatial_index(lockers_raw)

    prox_features = []
    for i, row in lockers_joined.iterrows():
        lat, lng = row["lat"], row["lng"]

        prox_features.append({
            "competitors_500m": count_within_radius(comp_idx, competitors_raw, lat, lng, 500),
            "competitors_1km": count_within_radius(comp_idx, competitors_raw, lat, lng, 1000),
            "supermarkets_500m": count_within_radius(super_idx, supermarkets_raw, lat, lng, 500),
            "supermarkets_1km": count_within_radius(super_idx, supermarkets_raw, lat, lng, 1000),
            "bpost_500m": count_within_radius(locker_idx, lockers_raw, lat, lng, 500) - 1,  # exclude self
            "bpost_1km": count_within_radius(locker_idx, lockers_raw, lat, lng, 1000) - 1,
            "bpost_2km": count_within_radius(locker_idx, lockers_raw, lat, lng, 2000) - 1,
            "dist_nearest_competitor": nearest_distance(comp_idx, competitors_raw, lat, lng),
            "dist_nearest_supermarket": nearest_distance(super_idx, supermarkets_raw, lat, lng),
            "dist_nearest_other_locker": nearest_distance(
                locker_idx, lockers_raw, lat, lng, exclude_idx=i),
        })

        if (i + 1) % 500 == 0:
            print(f"  Processed {i + 1}/{len(lockers_joined)} lockers")

    prox_df = pd.DataFrame(prox_features, index=lockers_joined.index)
    lockers_joined = pd.concat([lockers_joined, prox_df], axis=1)

    # Select columns for the enriched dataframe
    keep_cols = [
        "id", "lat", "lng", "name",
        # Sector assignment
        "sc", "mun", "region", "zone", "area",
        # Demographics
        "pop", "dens", "demand", "ageRatio", "incomeIdx",
        # Competition
        "cc", "oc", "gap", "num_operators",
        # Quadrant
        "quadrant",
        # Municipality (for CV)
        "municipality",
        # Proximity
        "competitors_500m", "competitors_1km",
        "supermarkets_500m", "supermarkets_1km",
        "bpost_500m", "bpost_1km", "bpost_2km",
        "dist_nearest_competitor", "dist_nearest_supermarket",
        "dist_nearest_other_locker",
    ]
    lockers_df = lockers_joined[keep_cols].copy()

    # Save cache
    lockers_df.to_parquet(CACHE_LOCAL, index=False)
    print(f"\nSaved to {CACHE_LOCAL}")
    print(f"Shape: {lockers_df.shape}")

print(f"\nSample:\n{lockers_df.head(3).T}")

## Section 2: OSM Enrichment via Overpass Tile Batching (~200 API calls)

Divide Belgium into 0.2° x 0.2° tiles and fetch transit, commerce, parking, and pedestrian features from Overpass in a single query per tile. Then assign proximity counts to each locker.

In [ ]:
CACHE_OSM_RAW = ENRICHED / "osm_raw_elements.json"
CACHE_OSM = ENRICHED / "osm_tile_features.parquet"

OVERPASS_URL = "https://overpass-api.de/api/interpreter"
HEADERS = {"User-Agent": "bbox-coverage-tool/1.0 (locker-analysis)"}

# Belgium bounding box
BE_SOUTH, BE_NORTH = 49.5, 51.55
BE_WEST, BE_EAST = 2.5, 6.45
TILE_SIZE_DEG = 0.2


def overpass_query(query, retries=3):
    """POST query to Overpass with retry + exponential backoff."""
    for attempt in range(retries):
        try:
            resp = requests.post(OVERPASS_URL, data={"data": query},
                                 headers=HEADERS, timeout=120)
            resp.raise_for_status()
            return resp.json()
        except Exception as e:
            if attempt < retries - 1:
                wait = 5 * (attempt + 1)
                print(f"    Overpass query failed ({e}), retrying in {wait}s...")
                time.sleep(wait)
            else:
                print(f"    Overpass query failed after {retries} attempts: {e}")
                return {"elements": []}


def build_tile_query(south, west, north, east):
    """Build a single Overpass query that fetches all feature categories for a tile."""
    bbox = f"{south:.4f},{west:.4f},{north:.4f},{east:.4f}"
    return f"""[out:json][timeout:90][bbox:{bbox}];
(
  node["highway"="bus_stop"];
  node["railway"~"station|halt|tram_stop"];
  nwr["shop"];
  nwr["amenity"~"restaurant|cafe|bank|pharmacy|post_office"];
  nwr["amenity"="parking"];
  way["highway"="footway"];
);
out center body;"""


def classify_osm_element(el):
    """Classify an OSM element into a feature category."""
    tags = el.get("tags", {})
    if tags.get("highway") == "bus_stop":
        return "bus_stop"
    if tags.get("railway") in ("station", "halt", "tram_stop"):
        return "rail_tram"
    if "shop" in tags:
        return "shop"
    if tags.get("amenity") in ("restaurant", "cafe", "bank", "pharmacy", "post_office"):
        return "amenity"
    if tags.get("amenity") == "parking":
        return "parking"
    if tags.get("highway") == "footway":
        return "footway"
    return None


def get_element_coords(el):
    """Extract lat/lng from an OSM element (node or way with center)."""
    lat = el.get("lat") or (el.get("center", {}).get("lat"))
    lng = el.get("lon") or (el.get("center", {}).get("lon"))
    return lat, lng


if CACHE_OSM.exists():
    print(f"Loading cached OSM features from {CACHE_OSM}")
    osm_features_df = pd.read_parquet(CACHE_OSM)
    print(f"  {len(osm_features_df)} rows loaded")
else:
    # --- 2a. Generate tile grid ---
    tiles = []
    lat = BE_SOUTH
    while lat < BE_NORTH:
        lng = BE_WEST
        while lng < BE_EAST:
            tiles.append((lat, lng, min(lat + TILE_SIZE_DEG, BE_NORTH),
                          min(lng + TILE_SIZE_DEG, BE_EAST)))
            lng += TILE_SIZE_DEG
        lat += TILE_SIZE_DEG

    print(f"2a. Generated {len(tiles)} tiles ({TILE_SIZE_DEG}° × {TILE_SIZE_DEG}°)")

    # --- 2b. Fetch all tiles (with caching of raw elements) ---
    if CACHE_OSM_RAW.exists():
        print(f"Loading cached raw OSM elements from {CACHE_OSM_RAW}")
        with open(CACHE_OSM_RAW) as f:
            all_elements_by_cat = json.load(f)
        total = sum(len(v) for v in all_elements_by_cat.values())
        print(f"  {total} elements across {len(all_elements_by_cat)} categories")
    else:
        all_elements_by_cat = {
            "bus_stop": [], "rail_tram": [], "shop": [],
            "amenity": [], "parking": [], "footway": [],
        }
        seen_ids = set()

        for ti, (south, west, north, east) in enumerate(tiles):
            query = build_tile_query(south, west, north, east)
            result = overpass_query(query)

            added = 0
            for el in result.get("elements", []):
                eid = el.get("id")
                if eid in seen_ids:
                    continue
                seen_ids.add(eid)

                cat = classify_osm_element(el)
                if cat is None:
                    continue

                lat_e, lng_e = get_element_coords(el)
                if lat_e is None or lng_e is None:
                    continue

                all_elements_by_cat[cat].append({"lat": lat_e, "lng": lng_e})
                added += 1

            if (ti + 1) % 20 == 0 or ti == len(tiles) - 1:
                total = sum(len(v) for v in all_elements_by_cat.values())
                print(f"  Tile {ti + 1}/{len(tiles)}: +{added} elements, "
                      f"total={total}")

            time.sleep(1)  # Rate limit

        # Save raw elements
        with open(CACHE_OSM_RAW, "w") as f:
            json.dump(all_elements_by_cat, f)
        total = sum(len(v) for v in all_elements_by_cat.values())
        print(f"\nSaved {total} raw OSM elements to {CACHE_OSM_RAW}")

    # --- 2c. Build spatial indices per category ---
    print("2c. Building spatial indices for OSM categories...")
    osm_indices = {}
    for cat, elements in all_elements_by_cat.items():
        osm_indices[cat] = build_spatial_index(elements)
        print(f"  {cat}: {len(elements)} elements")

    # --- 2d. Compute proximity features for each locker ---
    print("2d. Computing OSM proximity features for lockers...")
    osm_prox = []
    for i, row in lockers_df.iterrows():
        lat, lng = row["lat"], row["lng"]
        osm_prox.append({
            "bus_stops_300m": count_within_radius(
                osm_indices["bus_stop"], all_elements_by_cat["bus_stop"], lat, lng, 300),
            "rail_tram_300m": count_within_radius(
                osm_indices["rail_tram"], all_elements_by_cat["rail_tram"], lat, lng, 300),
            "shops_300m": count_within_radius(
                osm_indices["shop"], all_elements_by_cat["shop"], lat, lng, 300),
            "amenities_300m": count_within_radius(
                osm_indices["amenity"], all_elements_by_cat["amenity"], lat, lng, 300),
            "parking_500m": count_within_radius(
                osm_indices["parking"], all_elements_by_cat["parking"], lat, lng, 500),
            "footways_300m": count_within_radius(
                osm_indices["footway"], all_elements_by_cat["footway"], lat, lng, 300),
        })

    osm_features_df = pd.DataFrame(osm_prox)

    # Save cache
    osm_features_df.to_parquet(CACHE_OSM, index=False)
    print(f"\nSaved OSM features to {CACHE_OSM}")
    print(f"Shape: {osm_features_df.shape}")

print(f"\nOSM feature summary:\n{osm_features_df.describe().round(1)}")

## Section 3: Zoning Enrichment (~2,400 API calls)

Query regional zoning APIs for each locker to classify its urban planning zone:
- **Brussels** (BruGIS WFS): PRAS zoning (Zones mixtes, Zones d'habitation, etc.)
- **Wallonia** (SPW ArcGIS): Plan de secteur
- **Flanders** (Geopunt WFS): Gewestplan

API patterns adapted from `scripts/local_analysis.py` lines 873-957.

In [ ]:
CACHE_ZONING = ENRICHED / "zoning_features.parquet"

ZONING_HEADERS = {"User-Agent": "bbox-coverage-tool/1.0 (locker-analysis)"}

# Pre-check: is the Flanders gewestplan WFS alive?
_FLANDERS_API_ALIVE = False
try:
    _test = requests.get(
        "https://geo.api.vlaanderen.be/gewestplan/wfs?SERVICE=WFS&REQUEST=GetCapabilities",
        timeout=5)
    _FLANDERS_API_ALIVE = _test.status_code == 200 and "FeatureType" in _test.text
except Exception:
    pass
print(f"Flanders gewestplan WFS available: {_FLANDERS_API_ALIVE}")


def _query_brussels_pras(lat, lng, retries=3):
    """Query PRAS zoning at coordinate via BruGIS WFS."""
    dlat, dlng = 0.001, 0.002
    bbox = f"{lng-dlng},{lat-dlat},{lng+dlng},{lat+dlat}"
    url = (f"https://gis.urban.brussels/geoserver/ows?"
           f"SERVICE=WFS&VERSION=2.0.0&REQUEST=GetFeature"
           f"&TYPENAME=PERSPECTIVE_FR:Affectations"
           f"&BBOX={bbox},EPSG:4326"
           f"&outputFormat=application/json&COUNT=3")
    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=ZONING_HEADERS, timeout=15)
            data = resp.json()
            features = data.get("features", [])
            if features:
                props = features[0].get("properties", {})
                return props.get("AFFECTATION", "unknown")
            return "no_data"
        except Exception:
            if attempt < retries - 1:
                time.sleep(2 * (attempt + 1))
            else:
                return "error"


def _query_wallonia_pds(lat, lng, retries=3):
    """Query Plan de secteur at coordinate via SPW ArcGIS REST."""
    url = (f"https://geoservices.wallonie.be/arcgis/rest/services/"
           f"AMENAGEMENT_TERRITOIRE/PDS/MapServer/identify?"
           f"geometry={lng},{lat}&geometryType=esriGeometryPoint&sr=4326"
           f"&layers=all&tolerance=5&mapExtent=0,0,1,1&imageDisplay=100,100,96"
           f"&returnGeometry=false&f=json")
    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=ZONING_HEADERS, timeout=15)
            data = resp.json()
            results = data.get("results", [])
            for r in results:
                layer = r.get("layerName", "")
                attrs = r.get("attributes", {})
                if "Secteurs" in layer:
                    return attrs.get("NOM", "unknown")
            return "no_secteur" if results else "no_data"
        except Exception:
            if attempt < retries - 1:
                time.sleep(2 * (attempt + 1))
            else:
                return "error"


def _query_flanders_gewestplan(lat, lng, retries=3):
    """Query Gewestplan at coordinate via Geopunt WFS."""
    if not _FLANDERS_API_ALIVE:
        return "api_unavailable"
    dlat, dlng = 0.001, 0.002
    bbox = f"{lng-dlng},{lat-dlat},{lng+dlng},{lat+dlat}"
    url = (f"https://geo.api.vlaanderen.be/gewestplan/wfs?"
           f"SERVICE=WFS&VERSION=2.0.0&REQUEST=GetFeature"
           f"&BBOX={bbox},EPSG:4326"
           f"&outputFormat=application/json&COUNT=3")
    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=ZONING_HEADERS, timeout=15)
            data = resp.json()
            features = data.get("features", [])
            if features:
                props = features[0].get("properties", {})
                return props.get("BESTTYP", props.get("type", "unknown"))
            return "no_data"
        except Exception:
            if attempt < retries - 1:
                time.sleep(2 * (attempt + 1))
            else:
                return "error"


def query_zoning(lat, lng, region):
    """Route to the correct regional zoning API."""
    if region == "Brussels-Capital":
        return _query_brussels_pras(lat, lng)
    elif region == "Wallonia":
        return _query_wallonia_pds(lat, lng)
    elif region == "Flanders":
        return _query_flanders_gewestplan(lat, lng)
    return "unknown_region"


if CACHE_ZONING.exists():
    print(f"Loading cached zoning features from {CACHE_ZONING}")
    zoning_df = pd.read_parquet(CACHE_ZONING)
    print(f"  {len(zoning_df)} rows loaded")
else:
    print("3. Querying zoning APIs for each locker...")
    print(f"   Flanders lockers will return 'api_unavailable' (WFS is down)")
    zoning_results = []
    errors = 0

    for i, row in lockers_df.iterrows():
        zone_type = query_zoning(row["lat"], row["lng"], row["region"])
        zoning_results.append({"zoning_type": zone_type})

        if zone_type == "error":
            errors += 1

        if (i + 1) % 200 == 0:
            print(f"  {i + 1}/{len(lockers_df)} queried "
                  f"({errors} errors so far)")

        # Only rate-limit for live API calls
        if row["region"] != "Flanders" or _FLANDERS_API_ALIVE:
            time.sleep(0.5)

    zoning_df = pd.DataFrame(zoning_results)

    # Save cache
    zoning_df.to_parquet(CACHE_ZONING, index=False)
    print(f"\nSaved zoning features to {CACHE_ZONING}")
    print(f"Errors: {errors}/{len(lockers_df)}")

print(f"\nZoning type distribution (top 15):")
print(zoning_df["zoning_type"].value_counts().head(15).to_string())

## Section 4: Negative Sample Generation & Training Dataset Assembly

Generate ~7,000 negative samples (locations where bpost chose NOT to place lockers):
1. **Hard negatives (~2,500):** High-demand sectors with no locker within 2km
2. **Random negatives (~3,000):** Randomly sampled sectors, stratified by zone type
3. **Near-miss negatives (~1,500):** Sectors adjacent to locker sectors but without lockers

Then enrich all negatives with the same features as positives and combine into a training dataset.

In [ ]:
CACHE_TRAINING = TRAINING / "training_dataset.parquet"
CACHE_OSM_RAW = ENRICHED / "osm_raw_elements.json"

if CACHE_TRAINING.exists():
    print(f"Loading cached training dataset from {CACHE_TRAINING}")
    training_df = pd.read_parquet(CACHE_TRAINING)
    print(f"  {len(training_df)} samples ({training_df['label'].sum()} positive, "
          f"{(training_df['label'] == 0).sum()} negative)")
else:
    np.random.seed(42)

    # Ensure spatial indices are available (may have been skipped by cache in earlier sections)
    if "comp_idx" not in dir():
        print("Building spatial indices (skipped by cache earlier)...")
        comp_idx = build_spatial_index(competitors_raw)
        super_idx = build_spatial_index(supermarkets_raw)
        locker_idx = build_spatial_index(lockers_raw)

    if "osm_indices" not in dir():
        if CACHE_OSM_RAW.exists():
            with open(CACHE_OSM_RAW) as f:
                all_elements_by_cat = json.load(f)
            osm_indices = {}
            for cat, elements in all_elements_by_cat.items():
                osm_indices[cat] = build_spatial_index(elements)
        else:
            print("  WARNING: No OSM data available. OSM features will be 0.")
            all_elements_by_cat = {k: [] for k in
                                   ["bus_stop", "rail_tram", "shop",
                                    "amenity", "parking", "footway"]}
            osm_indices = {k: {} for k in all_elements_by_cat}

    # --- 4a. Identify sectors with lockers ---
    locker_sectors = set(lockers_df["sc"].dropna().unique())
    print(f"4a. Sectors with lockers: {len(locker_sectors)}")

    all_sector_codes = set(sectors_gdf["sc"].values)
    non_locker_sectors = all_sector_codes - locker_sectors
    print(f"    Sectors without lockers: {len(non_locker_sectors)}")

    # Prepare sector lookup for fast enrichment
    sector_props = {}
    for _, row in sectors_gdf.iterrows():
        sc = row["sc"]
        sector_props[sc] = {
            "sc": sc, "mun": row["mun"], "region": row["region"],
            "zone": row["zone"], "area": row["area"],
            "pop": row["pop"], "dens": row["dens"],
            "lat": row["clat"], "lng": row["clng"],
        }

    # --- 4b. Hard negatives: high-demand sectors without lockers ---
    print("4b. Generating hard negatives...")

    # Get demand values for all sectors
    demand_values = []
    for sc in non_locker_sectors:
        d = centroids_map.get(sc)
        if d:
            demand_values.append((sc, d["demand"]))

    demand_values.sort(key=lambda x: x[1], reverse=True)
    demand_75th = np.percentile([d for _, d in demand_values], 75)
    print(f"    Demand 75th percentile: {demand_75th:.1f}")

    # Filter: high demand AND no locker within 2km of centroid
    hard_negatives = []
    for sc, demand in demand_values:
        if demand < demand_75th:
            continue
        props = sector_props.get(sc)
        if not props:
            continue
        lat, lng = props["lat"], props["lng"]
        nearby_lockers = count_within_radius(locker_idx, lockers_raw, lat, lng, 2000)
        if nearby_lockers == 0:
            hard_negatives.append(sc)
        if len(hard_negatives) >= 2500:
            break

    print(f"    Hard negatives: {len(hard_negatives)}")

    # --- 4c. Near-miss negatives: adjacent to locker sectors ---
    print("4c. Generating near-miss negatives...")

    # Find sectors that touch locker-containing sectors
    locker_gdf = sectors_gdf[sectors_gdf["sc"].isin(locker_sectors)]
    non_locker_gdf = sectors_gdf[sectors_gdf["sc"].isin(
        non_locker_sectors - set(hard_negatives))]

    # Use spatial join with 'touches' predicate
    adjacent = gpd.sjoin(non_locker_gdf, locker_gdf, how="inner",
                         predicate="touches")
    near_miss_scs = list(adjacent["sc_left"].unique())
    np.random.shuffle(near_miss_scs)
    near_miss_negatives = near_miss_scs[:1500]
    print(f"    Near-miss negatives: {len(near_miss_negatives)} "
          f"(from {len(near_miss_scs)} adjacent sectors)")

    # --- 4d. Random negatives: stratified by zone type ---
    print("4d. Generating random negatives...")
    used_sectors = locker_sectors | set(hard_negatives) | set(near_miss_negatives)
    remaining = non_locker_sectors - used_sectors

    # Stratify by zone type
    remaining_by_zone = defaultdict(list)
    for sc in remaining:
        props = sector_props.get(sc)
        if props:
            remaining_by_zone[props["zone"]].append(sc)

    # Target 3000, proportional to zone distribution
    total_remaining = sum(len(v) for v in remaining_by_zone.values())
    random_negatives = []
    for zone, scs in remaining_by_zone.items():
        n_sample = int(3000 * len(scs) / total_remaining)
        n_sample = min(n_sample, len(scs))
        random_negatives.extend(
            np.random.choice(scs, size=n_sample, replace=False).tolist())

    print(f"    Random negatives: {len(random_negatives)}")

    # --- 4e. Combine all negatives and enrich ---
    print("4e. Enriching negative samples...")

    all_negative_scs = hard_negatives + near_miss_negatives + random_negatives
    neg_types = (["hard"] * len(hard_negatives) +
                 ["near_miss"] * len(near_miss_negatives) +
                 ["random"] * len(random_negatives))

    neg_records = []
    for idx, (sc, neg_type) in enumerate(zip(all_negative_scs, neg_types)):
        props = sector_props.get(sc, {})
        lat = props.get("lat", 0)
        lng = props.get("lng", 0)

        centroid_data = centroids_map.get(sc, {})

        record = {
            "lat": lat, "lng": lng,
            "sc": sc,
            "mun": props.get("mun", ""),
            "region": props.get("region", ""),
            "zone": props.get("zone", ""),
            "area": props.get("area", 0),
            "pop": props.get("pop", 0),
            "dens": props.get("dens", 0),
            "demand": centroid_data.get("demand", 0),
            "ageRatio": centroid_data.get("ageRatio", 0),
            "incomeIdx": centroid_data.get("incomeIdx", 0),
            "cc": comp_cov_sectors.get(sc, {}).get("cc", 0),
            "oc": comp_cov_sectors.get(sc, {}).get("oc", 0),
            "gap": comp_cov_sectors.get(sc, {}).get("gap", 1.0),
            "num_operators": len(comp_cov_sectors.get(sc, {}).get("ops", [])),
            "quadrant": quadrants_sectors.get(sc, "unknown"),
            "municipality": demand_map.get(sc, {}).get("mun", "unknown"),
            # Proximity
            "competitors_500m": count_within_radius(
                comp_idx, competitors_raw, lat, lng, 500),
            "competitors_1km": count_within_radius(
                comp_idx, competitors_raw, lat, lng, 1000),
            "supermarkets_500m": count_within_radius(
                super_idx, supermarkets_raw, lat, lng, 500),
            "supermarkets_1km": count_within_radius(
                super_idx, supermarkets_raw, lat, lng, 1000),
            "bpost_500m": count_within_radius(
                locker_idx, lockers_raw, lat, lng, 500),
            "bpost_1km": count_within_radius(
                locker_idx, lockers_raw, lat, lng, 1000),
            "bpost_2km": count_within_radius(
                locker_idx, lockers_raw, lat, lng, 2000),
            "dist_nearest_competitor": nearest_distance(
                comp_idx, competitors_raw, lat, lng),
            "dist_nearest_supermarket": nearest_distance(
                super_idx, supermarkets_raw, lat, lng),
            "dist_nearest_other_locker": nearest_distance(
                locker_idx, lockers_raw, lat, lng),
            # OSM
            "bus_stops_300m": count_within_radius(
                osm_indices["bus_stop"], all_elements_by_cat["bus_stop"],
                lat, lng, 300),
            "rail_tram_300m": count_within_radius(
                osm_indices["rail_tram"], all_elements_by_cat["rail_tram"],
                lat, lng, 300),
            "shops_300m": count_within_radius(
                osm_indices["shop"], all_elements_by_cat["shop"],
                lat, lng, 300),
            "amenities_300m": count_within_radius(
                osm_indices["amenity"], all_elements_by_cat["amenity"],
                lat, lng, 300),
            "parking_500m": count_within_radius(
                osm_indices["parking"], all_elements_by_cat["parking"],
                lat, lng, 500),
            "footways_300m": count_within_radius(
                osm_indices["footway"], all_elements_by_cat["footway"],
                lat, lng, 300),
            # Zoning (sector-level proxy for negatives)
            "zoning_type": "unknown",
            # Meta
            "label": 0,
            "neg_type": neg_type,
        }
        neg_records.append(record)

        if (idx + 1) % 1000 == 0:
            print(f"  Enriched {idx + 1}/{len(all_negative_scs)} negatives")

    neg_df = pd.DataFrame(neg_records)
    print(f"\nNegative samples: {len(neg_df)}")
    print(f"  hard: {(neg_df['neg_type'] == 'hard').sum()}")
    print(f"  near_miss: {(neg_df['neg_type'] == 'near_miss').sum()}")
    print(f"  random: {(neg_df['neg_type'] == 'random').sum()}")

    # --- 4f. Combine positives and negatives ---
    print("\n4f. Assembling training dataset...")

    # Build positive records
    pos_df = lockers_df.copy()
    pos_df = pd.concat([pos_df.reset_index(drop=True),
                        osm_features_df.reset_index(drop=True),
                        zoning_df.reset_index(drop=True)], axis=1)
    pos_df["label"] = 1
    pos_df["neg_type"] = "positive"

    # Align columns
    for c in neg_df.columns:
        if c not in pos_df.columns:
            pos_df[c] = 0
    for c in pos_df.columns:
        if c not in neg_df.columns:
            neg_df[c] = np.nan

    training_df = pd.concat([pos_df, neg_df], ignore_index=True)

    # Replace inf with large value for distance features
    dist_cols = [c for c in training_df.columns if c.startswith("dist_")]
    for c in dist_cols:
        training_df[c] = training_df[c].replace([np.inf], 50000)

    # Save
    training_df.to_parquet(CACHE_TRAINING, index=False)
    print(f"\nSaved training dataset to {CACHE_TRAINING}")

print(f"\nTraining dataset: {len(training_df)} samples")
print(f"  Positives: {(training_df['label'] == 1).sum()}")
print(f"  Negatives: {(training_df['label'] == 0).sum()}")
print(f"  Ratio: 1:{(training_df['label'] == 0).sum() / (training_df['label'] == 1).sum():.1f}")

## Section 4.5: Exploratory Data Analysis

Visual inspection of the training dataset before model training:
- Class balance and negative sample type breakdown
- Feature distributions (positives vs negatives)
- Correlation heatmap (flag multicollinearity)
- Spatial distribution of samples

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class balance
counts = training_df["label"].value_counts()
axes[0].bar(["Negative (0)", "Positive (1)"], [counts[0], counts[1]],
            color=["#e74c3c", "#2ecc71"])
axes[0].set_title("Class Balance")
axes[0].set_ylabel("Count")
for i, v in enumerate([counts[0], counts[1]]):
    axes[0].text(i, v + 50, str(v), ha="center", fontweight="bold")

# Negative type breakdown
neg_counts = training_df[training_df["label"] == 0]["neg_type"].value_counts()
axes[1].bar(neg_counts.index, neg_counts.values, color=["#e67e22", "#9b59b6", "#3498db"])
axes[1].set_title("Negative Sample Types")
axes[1].set_ylabel("Count")
for i, v in enumerate(neg_counts.values):
    axes[1].text(i, v + 50, str(v), ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig(OUTPUT / "class_balance.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved class_balance.png")

In [ ]:
# Feature distributions: positives vs negatives
key_features = ["pop", "dens", "demand", "competitors_500m", "supermarkets_500m",
                "bus_stops_300m", "shops_300m", "dist_nearest_competitor"]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    if feat not in training_df.columns:
        continue
    pos_vals = training_df.loc[training_df["label"] == 1, feat].dropna()
    neg_vals = training_df.loc[training_df["label"] == 0, feat].dropna()

    axes[i].hist(neg_vals, bins=40, alpha=0.5, label="Negative", color="#e74c3c",
                 density=True)
    axes[i].hist(pos_vals, bins=40, alpha=0.5, label="Positive", color="#2ecc71",
                 density=True)
    axes[i].set_title(feat)
    axes[i].legend(fontsize=8)

plt.suptitle("Feature Distributions: Positive vs Negative", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT / "feature_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved feature_distributions.png")

In [ ]:
# Correlation heatmap for numerical features
NUMERIC_FEATURES = [
    "pop", "dens", "demand", "ageRatio", "incomeIdx", "area",
    "cc", "oc", "gap", "num_operators",
    "competitors_500m", "competitors_1km",
    "supermarkets_500m", "supermarkets_1km",
    "bpost_500m", "bpost_1km", "bpost_2km",
    "dist_nearest_competitor", "dist_nearest_supermarket",
    "dist_nearest_other_locker",
    "bus_stops_300m", "rail_tram_300m", "shops_300m",
    "amenities_300m", "parking_500m", "footways_300m",
]
available_numeric = [f for f in NUMERIC_FEATURES if f in training_df.columns]

corr = training_df[available_numeric].corr()

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, ax=ax, square=True, linewidths=0.5)
ax.set_title("Feature Correlation Matrix", fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT / "correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

# Flag highly correlated pairs
high_corr_pairs = []
for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.8:
            high_corr_pairs.append(
                (corr.columns[i], corr.columns[j], corr.iloc[i, j]))

if high_corr_pairs:
    print("Highly correlated feature pairs (|r| > 0.8):")
    for f1, f2, r in sorted(high_corr_pairs, key=lambda x: -abs(x[2])):
        print(f"  {f1} <-> {f2}: r={r:.3f}")
else:
    print("No feature pairs with |r| > 0.8")

In [ ]:
# Spatial distribution of samples
fig, ax = plt.subplots(figsize=(10, 12))

neg = training_df[training_df["label"] == 0]
pos = training_df[training_df["label"] == 1]

ax.scatter(neg["lng"], neg["lat"], s=1, alpha=0.3, c="#e74c3c", label="Negative")
ax.scatter(pos["lng"], pos["lat"], s=3, alpha=0.5, c="#2ecc71", label="Positive")

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Spatial Distribution of Training Samples")
ax.legend(markerscale=5)
ax.set_aspect(1.5)  # Approximate aspect ratio at Belgium's latitude
plt.tight_layout()
plt.savefig(OUTPUT / "spatial_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved spatial_distribution.png")

## Section 5: XGBoost Binary Classifier with Spatial Cross-Validation

Train a gradient-boosted tree classifier with:
- GroupKFold (5 folds) by municipality to prevent spatial leakage
- `scale_pos_weight` to handle class imbalance
- Metrics: AUC-ROC, precision, recall, F1 per fold

In [ ]:
# --- 5a. Prepare feature matrix ---

# Preserve full dataset, but train only on samples with complete data.
# Flanders zoning API is down, so exclude Flanders lockers and their negatives.
# Also drop zoning_encoded — negatives have "unknown" zoning regardless, so it leaks.

complete_regions = ["Brussels-Capital", "Wallonia"]
mask = training_df["region"].isin(complete_regions)
train_subset = training_df[mask].copy().reset_index(drop=True)

n_pos = (train_subset["label"] == 1).sum()
n_neg = (train_subset["label"] == 0).sum()
print(f"Filtered to complete-data regions: {complete_regions}")
print(f"  Full dataset preserved: {len(training_df)} samples")
print(f"  Training subset: {len(train_subset)} samples "
      f"({n_pos} positive, {n_neg} negative)")

# Numerical features
FEATURE_COLS = [
    # Demographics
    "pop", "dens", "demand", "ageRatio", "incomeIdx", "area",
    # Competition
    "cc", "oc", "gap", "num_operators",
    # Proximity counts
    "competitors_500m", "competitors_1km",
    "supermarkets_500m", "supermarkets_1km",
    "bpost_500m", "bpost_1km", "bpost_2km",
    # Distances
    "dist_nearest_competitor", "dist_nearest_supermarket",
    "dist_nearest_other_locker",
    # OSM
    "bus_stops_300m", "rail_tram_300m", "shops_300m",
    "amenities_300m", "parking_500m", "footways_300m",
]

# Filter to available columns
feature_cols = [c for c in FEATURE_COLS if c in train_subset.columns]
print(f"Numerical features: {len(feature_cols)}")

# One-hot encode categorical features
zone_dummies = pd.get_dummies(train_subset["zone"], prefix="zone")
quad_dummies = pd.get_dummies(train_subset["quadrant"], prefix="quad")

# NOTE: zoning_encoded deliberately excluded — it leaks label info
# (positives have real zoning types, negatives always "unknown")

# Build X matrix
X = pd.concat([
    train_subset[feature_cols].fillna(0),
    zone_dummies,
    quad_dummies,
], axis=1)

y = train_subset["label"].values
groups = train_subset["municipality"].fillna("unknown").values

all_feature_names = list(X.columns)
print(f"Total features: {len(all_feature_names)}")
print(f"Samples: {len(X)} (positive: {y.sum()}, negative: {(y == 0).sum()})")
print(f"Unique groups (municipalities): {len(set(groups))}")

In [ ]:
# --- 5b. Spatial cross-validation with GroupKFold ---

n_neg = (y == 0).sum()
n_pos = y.sum()
spw = n_neg / n_pos if n_pos > 0 else 1.0
print(f"scale_pos_weight: {spw:.2f}")

gkf = GroupKFold(n_splits=5)

fold_metrics = []
all_test_indices = []
all_test_preds = []
all_test_true = []

X_np = X.values.astype(np.float32)

for fold, (train_idx, test_idx) in enumerate(gkf.split(X_np, y, groups)):
    X_train, X_test = X_np[train_idx], X_np[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=spw,
        eval_metric="auc",
        random_state=42,
        verbosity=0,
    )
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba >= 0.5).astype(int)

    auc = roc_auc_score(y_test, y_pred_proba)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    fold_metrics.append({"fold": fold + 1, "auc_roc": auc, "precision": prec,
                         "recall": rec, "f1": f1,
                         "n_train": len(train_idx), "n_test": len(test_idx)})

    all_test_indices.extend(test_idx)
    all_test_preds.extend(y_pred_proba)
    all_test_true.extend(y_test)

    print(f"  Fold {fold + 1}: AUC={auc:.4f}  Prec={prec:.4f}  "
          f"Rec={rec:.4f}  F1={f1:.4f}  "
          f"(train={len(train_idx)}, test={len(test_idx)})")

metrics_df = pd.DataFrame(fold_metrics)
print(f"\nMean metrics across {len(fold_metrics)} folds:")
print(f"  AUC-ROC:   {metrics_df['auc_roc'].mean():.4f} +/- {metrics_df['auc_roc'].std():.4f}")
print(f"  Precision: {metrics_df['precision'].mean():.4f} +/- {metrics_df['precision'].std():.4f}")
print(f"  Recall:    {metrics_df['recall'].mean():.4f} +/- {metrics_df['recall'].std():.4f}")
print(f"  F1:        {metrics_df['f1'].mean():.4f} +/- {metrics_df['f1'].std():.4f}")

In [ ]:
# --- 5c. Train final model on full dataset ---
print("5c. Training final model on full dataset...")

final_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=spw,
    eval_metric="auc",
    random_state=42,
    verbosity=0,
)
final_model.fit(X_np, y)
final_model.get_booster().feature_names = all_feature_names

print(f"Final model trained on {len(X_np)} samples with {len(all_feature_names)} features")

## Section 6: SHAP Analysis & Placement Profiles

Use SHAP to explain the model's predictions:
- Beeswarm plot: how each feature pushes predictions up or down
- Bar plot: mean absolute SHAP values (overall feature importance)
- Placement profiles: median feature values per zone type for positive samples

In [ ]:
# --- 6a. SHAP analysis ---
print("6a. Computing SHAP values...")

explainer = shap.TreeExplainer(final_model)

# Use a sample for SHAP if dataset is large (for speed)
if len(X_np) > 5000:
    shap_sample_idx = np.random.choice(len(X_np), 5000, replace=False)
    X_shap = X_np[shap_sample_idx]
else:
    X_shap = X_np

shap_values = explainer.shap_values(X_shap)

# Create a DataFrame for plotting
X_shap_df = pd.DataFrame(X_shap, columns=all_feature_names)

print(f"SHAP values computed for {len(X_shap)} samples")

In [ ]:
# Beeswarm plot
fig, ax = plt.subplots(figsize=(12, 10))
shap.summary_plot(shap_values, X_shap_df, max_display=20, show=False)
plt.tight_layout()
plt.savefig(OUTPUT / "shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved shap_beeswarm.png")

In [ ]:
# Bar plot (mean |SHAP|)
fig, ax = plt.subplots(figsize=(12, 10))
shap.summary_plot(shap_values, X_shap_df, plot_type="bar", max_display=20, show=False)
plt.tight_layout()
plt.savefig(OUTPUT / "shap_bar.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved shap_bar.png")

In [ ]:
# --- 6b. Feature importance table ---
print("6b. Building feature importance table...")

# XGBoost gain importance
xgb_importance = final_model.get_booster().get_score(importance_type="gain")

# SHAP mean absolute values
mean_shap = np.abs(shap_values).mean(axis=0)
shap_importance = dict(zip(all_feature_names, mean_shap))

# Combine
importance_records = []
for feat in all_feature_names:
    importance_records.append({
        "feature": feat,
        "xgb_gain": xgb_importance.get(feat, 0),
        "mean_abs_shap": shap_importance.get(feat, 0),
    })

importance_df = pd.DataFrame(importance_records)
importance_df = importance_df.sort_values("mean_abs_shap", ascending=False)
importance_df.to_csv(OUTPUT / "feature_importance.csv", index=False)

print(f"\nTop 20 features by mean |SHAP|:")
print(importance_df.head(20).to_string(index=False))

In [ ]:
# --- 6c. Placement profiles per zone type ---
print("6c. Building placement profiles per zone type...")

# Use only positive samples from the training subset
pos_data = train_subset[train_subset["label"] == 1].copy()

# Top features by importance (exclude one-hot and encoded)
top_features = importance_df[
    ~importance_df["feature"].str.startswith(("zone_", "quad_", "zoning_"))
].head(10)["feature"].tolist()

# Compute median and IQR per zone
profile_records = []
for zone in ["urban", "suburban", "rural"]:
    zone_data = pos_data[pos_data["zone"] == zone]
    if len(zone_data) == 0:
        continue
    record = {"zone": zone, "count": len(zone_data)}
    for feat in top_features:
        if feat in zone_data.columns:
            vals = zone_data[feat].dropna()
            record[f"{feat}_median"] = vals.median()
            record[f"{feat}_q25"] = vals.quantile(0.25)
            record[f"{feat}_q75"] = vals.quantile(0.75)
    profile_records.append(record)

profiles_df = pd.DataFrame(profile_records)
profiles_df.to_csv(OUTPUT / "zone_profiles.csv", index=False)

print(f"\nPlacement profiles ({len(top_features)} top features):")
print(profiles_df.to_string(index=False))

## Section 7: Save Model & Summary

In [ ]:
import joblib

# Save trained model
MODEL_PATH = TRAINING / "locker_placement_model.joblib"
joblib.dump({
    "model": final_model,
    "feature_names": all_feature_names,
    "metrics": metrics_df.to_dict(orient="records"),
    "scale_pos_weight": spw,
    "training_regions": ["Brussels-Capital", "Wallonia"],
    "excluded_regions": ["Flanders"],
    "exclusion_reason": "Flanders gewestplan WFS API down",
}, MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")

# Summary
print("\n" + "=" * 60)
print("LOCKER PLACEMENT PATTERN ANALYSIS — SUMMARY")
print("=" * 60)

print(f"\nDataset:")
print(f"  Full dataset (preserved): {len(training_df)} samples")
print(f"  Training subset (complete data): {len(train_subset)} samples")
print(f"  Positive (locker sites): {(train_subset['label'] == 1).sum()}")
print(f"  Negative (non-locker sites): {(train_subset['label'] == 0).sum()}")
print(f"  Regions: Brussels-Capital, Wallonia (Flanders excluded — API down)")
print(f"  Zoning feature: excluded (leaks label info)")

print(f"\nModel Performance (5-fold spatial CV):")
print(f"  AUC-ROC:   {metrics_df['auc_roc'].mean():.4f} +/- {metrics_df['auc_roc'].std():.4f}")
print(f"  Precision: {metrics_df['precision'].mean():.4f} +/- {metrics_df['precision'].std():.4f}")
print(f"  Recall:    {metrics_df['recall'].mean():.4f} +/- {metrics_df['recall'].std():.4f}")
print(f"  F1:        {metrics_df['f1'].mean():.4f} +/- {metrics_df['f1'].std():.4f}")

print(f"\nTop 5 Features (by SHAP):")
top5 = importance_df.head(5)
for _, row in top5.iterrows():
    print(f"  {row['feature']}: mean|SHAP| = {row['mean_abs_shap']:.4f}")

print(f"\nKey Insights by Zone Type:")
for _, profile in profiles_df.iterrows():
    zone = profile["zone"]
    count = int(profile["count"])
    print(f"\n  {zone.upper()} ({count} lockers):")
    for feat in top_features[:5]:
        med_col = f"{feat}_median"
        if med_col in profile:
            print(f"    {feat}: median = {profile[med_col]:.1f}")

print(f"\nOutputs saved to:")
print(f"  Model: {MODEL_PATH}")
print(f"  Training data: {CACHE_TRAINING}")
print(f"  Feature importance: {OUTPUT / 'feature_importance.csv'}")
print(f"  Zone profiles: {OUTPUT / 'zone_profiles.csv'}")
print(f"  SHAP beeswarm: {OUTPUT / 'shap_beeswarm.png'}")
print(f"  SHAP bar: {OUTPUT / 'shap_bar.png'}")
print(f"  Class balance: {OUTPUT / 'class_balance.png'}")
print(f"  Correlation heatmap: {OUTPUT / 'correlation_heatmap.png'}")
print(f"  Spatial distribution: {OUTPUT / 'spatial_distribution.png'}")
print("=" * 60)